# Model Deteksi Anomali DDoS — Threshold + One-Class SVM

Notebook ini khusus menangani **model**: memuat dataset, membersihkan dan membagi data,
menentukan aturan Tahap 1, melatih One-Class SVM, mengevaluasi, lalu menyimpan artefak model.

Artefak yang dihasilkan langsung dipakai oleh aplikasi web pada folder induk proyek:

| Berkas keluaran | Dipakai untuk |
|---|---|
| `model/ddos_two_stage_model.pkl` | model yang dimuat sistem web saat dinyalakan |
| `model/metrics.json` | angka pada halaman Dasbor |
| `data/data_demo.csv`, `data/data_uji.csv` | berkas contoh pada halaman Deteksi |
| `artefak/*.csv`, `artefak/evaluasi_hybrid.png` | bahan penulisan Bab IV |

Setelah notebook ini selesai dijalankan, jalankan `python app.py` dari folder induk proyek.

**Pencegahan kebocoran data (data leakage)**

| Langkah | Sumber informasi |
|---|---|
| Penghapusan duplikat | dilakukan sebelum pembagian data |
| Filter korelasi fitur | data latih Benign |
| Parameter normalisasi | data latih Benign |
| Ambang Tahap 1 | data latih Benign |
| `nu`, `gamma`, ambang Tahap 2 | data validasi |
| Data uji | dipakai satu kali pada poin 6 dan 7 |

In [ ]:
import glob, json, os, time, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

RANDOM_STATE = 42

# --- Lokasi berkas -----------------------------------------------------------
# Notebook berada di <proyek>/notebook, sehingga artefak ditulis ke folder induk.
PROJECT  = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = Path(os.environ.get("DDOS_DATASET", PROJECT / "dataset"))  # berisi berkas .parquet
MODEL_DIR    = PROJECT / "model"
DATA_OUT_DIR = PROJECT / "data"
ARTIFACT_DIR = PROJECT / "artefak"
for d in (MODEL_DIR, DATA_OUT_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Dataset CSE-CIC-IDS2018 versi parquet, mis. dari kaggle.com/datasets/dhoogla/csecicids2018.
# Di Kaggle, ganti DATA_DIR menjadi Path("/kaggle/input/csecicids2018").

# --- Lingkup dan pembatasan sampel -------------------------------------------
SCOPE = "ddos"              # "ddos" = hanya DoS/DDoS | "all" = semua jenis serangan
MAX_BENIGN_PER_FILE = 3000
MAX_ATTACK_PER_FILE = 2000

BENIGN_TRAIN, BENIGN_VAL = 0.60, 0.20   # sisanya 0.20 untuk uji
ATTACK_VAL = 0.30                       # sisanya 0.70 untuk uji

STAGE1_MAX_FPR = 0.01       # anggaran FPR Tahap 1 (data validasi)
SYSTEM_MAX_FPR = 0.05       # anggaran FPR sistem saat kalibrasi ambang
FPR_BUDGETS = [0.01, 0.02, 0.05, 0.10]

TUNE_SAMPLE = 10_000
NU_GRID    = [0.001, 0.005, 0.01, 0.02]
GAMMA_GRID = [0.1, 0.3, 0.5, 1.0, 2.0, 5.0]

np.random.seed(RANDOM_STATE)
print("Folder dataset :", DATA_DIR)
print("Folder proyek  :", PROJECT)

**Fungsi bantu** yang dipakai di beberapa poin sekaligus.

In [ ]:
def is_benign(label):
    return str(label).strip().lower() == "benign"

def is_dos(label):
    """Mencakup label 'DoS attacks-...' dan 'DDoS/DDOS attack-...'."""
    return "dos" in str(label).lower()

def signed_log1p(X):
    """Meredam ekor distribusi, aman untuk nilai negatif (mis. Init Fwd Win Bytes = -1)."""
    X = np.asarray(X, dtype=np.float64)
    return np.sign(X) * np.log1p(np.abs(X))

def apply_stage1(data, st):
    """Terapkan aturan Tahap 1. True = ditandai serangan."""
    if st["type"] == "rate":
        mask = np.zeros(len(data), dtype=bool)
        for f, t in st["thresholds"].items():
            mask |= data[f].values > t
        return mask
    X = data[st["features"]].values
    return ((X < st["lo"]) | (X > st["hi"])).sum(axis=1) >= st["k"]

def choose_threshold(y, score, max_fpr, forced=None):
    """Ambang skor dengan F1 tertinggi, dengan syarat FPR <= max_fpr.
    forced = sampel yang sudah diputus serangan oleh Tahap 1."""
    forced = np.zeros(len(y), dtype=bool) if forced is None else forced
    best_t, best_f1 = None, -1.0
    fallback = (np.inf, None)
    for t in np.unique(np.quantile(score, np.linspace(0, 1, 2001))):
        pred = forced | (score >= t)
        fp = np.sum(pred & (y == 0))
        tn = np.sum(~pred & (y == 0))
        fpr = fp / (fp + tn)
        if fpr <= max_fpr:
            f1 = f1_score(y, pred, zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = t, f1
        elif fpr < fallback[0]:
            fallback = (fpr, t)
    return best_t if best_t is not None else fallback[1]

def evaluate(name, y, pred, score=None):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Model": name,
        "Accuracy":  accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall":    recall_score(y, pred, zero_division=0),
        "F1-Score":  f1_score(y, pred, zero_division=0),
        "FPR":       fp / (fp + tn),
        "ROC-AUC":   roc_auc_score(y, score) if score is not None else np.nan,
        "PR-AUC":    average_precision_score(y, score) if score is not None else np.nan,
    }

def as_percent(df, skip=("Model", "Desain", "Himpunan")):
    out = df.copy()
    for c in out.columns:
        if c not in skip:
            out[c] = out[c].map(lambda v: "-" if pd.isna(v) else f"{v*100:.2f}%")
    return out

# 1. LOAD DATASET & CLEANING

In [ ]:
print("[1/7] Memuat dataset, cleaning, dan sampling...\n")

file_paths = sorted(glob.glob(str(DATA_DIR / "*.parquet")))
assert file_paths, f"Tidak ada berkas .parquet di {DATA_DIR}"

sampled = []
for path in file_paths:
    tmp = pd.read_parquet(path)
    tmp.columns = tmp.columns.str.strip()
    tmp["Label"] = tmp["Label"].astype(str).str.strip()

    num_cols = tmp.select_dtypes(include=np.number).columns
    tmp[num_cols] = tmp[num_cols].replace([np.inf, -np.inf], np.nan)
    tmp = tmp.dropna().drop_duplicates()

    for label, grp in tmp.groupby("Label"):
        if is_benign(label):
            n = MAX_BENIGN_PER_FILE
        elif SCOPE == "ddos" and not is_dos(label):
            continue
        else:
            n = MAX_ATTACK_PER_FILE
        sampled.append(grp.sample(n=min(len(grp), n), random_state=RANDOM_STATE))
    del tmp

df = pd.concat(sampled, ignore_index=True)
n_dup = len(df)
# Duplikat lintas berkas dibuang SEBELUM pembagian data, agar baris identik
# tidak muncul di data latih dan data uji sekaligus.
df = df.drop_duplicates().reset_index(drop=True)
df["y"] = (~df["Label"].map(is_benign)).astype(int)   # 0 = Benign, 1 = serangan
# Dicor sebelum pembagian data agar seluruh subset mewarisi tipe yang sama.
num_cols = df.select_dtypes(include=np.number).columns.drop("y")
df[num_cols] = df[num_cols].astype("float64")

print(f"Duplikat lintas berkas dibuang : {n_dup - len(df)}")
print(f"Total data (SCOPE='{SCOPE}')   : {len(df)}\n")
print(df["Label"].value_counts().to_string())

# 2. STRATIFIED SAMPLING & SELEKSI FITUR

Pembagian data serangan dilakukan secara berstrata per jenis serangan.
Data latih berisi **Benign saja**, sesuai prinsip One-Class SVM sebagai metode semi-supervised.

In [ ]:
print("[2/7] Membagi data latih, validasi, dan uji...\n")

benign = df[df["y"] == 0]
attack = df[df["y"] == 1]

b_train, b_rest = train_test_split(benign, train_size=BENIGN_TRAIN, random_state=RANDOM_STATE)
b_val, b_test = train_test_split(
    b_rest, train_size=BENIGN_VAL / (1 - BENIGN_TRAIN), random_state=RANDOM_STATE)

# Stratifikasi per label; label dengan < 2 sampel langsung masuk data uji.
counts = attack["Label"].value_counts()
rare = attack[attack["Label"].isin(counts[counts < 2].index)]
common = attack.drop(rare.index)
a_val, a_test = train_test_split(
    common, train_size=ATTACK_VAL, stratify=common["Label"], random_state=RANDOM_STATE)
a_test = pd.concat([a_test, rare])

val  = pd.concat([b_val,  a_val]).reset_index(drop=True)
test = pd.concat([b_test, a_test]).reset_index(drop=True)
y_val, y_test = val["y"].values, test["y"].values

print(pd.DataFrame([
    {"Himpunan": "Data latih",   "Benign": len(b_train), "Serangan": 0,          "Total": len(b_train)},
    {"Himpunan": "Data validasi","Benign": len(b_val),   "Serangan": len(a_val), "Total": len(val)},
    {"Himpunan": "Data uji",     "Benign": len(b_test),  "Serangan": len(a_test),"Total": len(test)},
]).to_string(index=False))

Seleksi fitur: pemilihan berbasis domain, lalu pembuangan fitur berkorelasi tinggi (dihitung dari data latih Benign).

In [ ]:
CANDIDATE_FEATURES = [
    # laju dan volume
    "Flow Duration", "Flow Bytes/s", "Flow Packets/s", "Fwd Packets/s", "Bwd Packets/s",
    "Total Fwd Packets", "Total Backward Packets", "Fwd Packets Length Total", "Bwd Packets Length Total",
    # ukuran paket
    "Packet Length Min", "Packet Length Max", "Packet Length Mean", "Packet Length Std",
    "Fwd Packet Length Max", "Fwd Packet Length Mean", "Bwd Packet Length Max", "Bwd Packet Length Mean",
    "Avg Packet Size", "Down/Up Ratio",
    # header dan TCP window
    "Fwd Header Length", "Bwd Header Length", "Init Fwd Win Bytes", "Init Bwd Win Bytes",
    "Fwd Seg Size Min", "Fwd Act Data Packets",
    # waktu antar paket
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min", "Fwd IAT Mean", "Bwd IAT Mean",
    "Active Mean", "Idle Mean",
    # flag TCP
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
]

FEATURES = [f for f in CANDIDATE_FEATURES if f in df.columns]
missing = [f for f in CANDIDATE_FEATURES if f not in df.columns]

corr = pd.DataFrame(signed_log1p(b_train[FEATURES]), columns=FEATURES).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
dropped = [c for c in upper.columns if (upper[c] > 0.98).any()]
FEATURES = [f for f in FEATURES if f not in dropped]

print(f"Fitur kandidat tersedia : {len(FEATURES) + len(dropped)}")
if missing:
    print(f"Tidak ada di dataset    : {missing}")
print(f"Dibuang (r>0.98)        : {dropped}")
print(f"Fitur final             : {len(FEATURES)}")

Transformasi `signed_log1p` lalu normalisasi. Scaler di-*fit* hanya pada data latih Benign.

In [ ]:
preproc = Pipeline([
    ("log", FunctionTransformer(signed_log1p)),
    ("scaler", StandardScaler()),
])
X_train = preproc.fit_transform(b_train[FEATURES])
X_val   = preproc.transform(val[FEATURES])
X_test  = preproc.transform(test[FEATURES])

print(f"Dimensi data latih: {X_train.shape}")

# 3. TAHAP 1: THRESHOLD-BASED DETECTION

Dua rancangan ambang dibandingkan, keduanya diturunkan dari persentil data latih Benign:

- **Rancangan A** — ambang atas pada fitur laju; ditandai jika salah satu fitur melampaui ambang.
- **Rancangan B** — pita persentil pada seluruh fitur; setiap fitur di luar pita memberi satu suara,
  ditandai jika suara ≥ K.

Konfigurasi dipilih yang paling agresif dengan FPR pada Benign validasi ≤ `STAGE1_MAX_FPR`.
Flow yang ditandai di tahap ini **langsung diputuskan sebagai serangan** dan tidak diteruskan ke Tahap 2.

In [ ]:
print("[3/7] Menentukan dan menerapkan aturan Tahap 1...\n")

RATE_FEATURES = [f for f in ["Flow Packets/s", "Flow Bytes/s", "Fwd Packets/s", "Bwd Packets/s"]
                 if f in df.columns]
candidates = []

# Rancangan A
for q in [99.0, 99.5, 99.8, 99.9, 99.95, 99.99]:
    st = {"type": "rate", "name": f"A: laju > p{q}",
          "thresholds": {f: float(np.percentile(b_train[f], q)) for f in RATE_FEATURES}}
    if apply_stage1(val, st)[y_val == 0].mean() <= STAGE1_MAX_FPR:
        candidates.append(st)
        break

# Rancangan B
for lo, hi in [(0.5, 99.5), (0.1, 99.9), (0.05, 99.95), (0.01, 99.99)]:
    st = {"type": "band", "features": FEATURES, "k": 1,
          "lo": np.percentile(b_train[FEATURES].values, lo, axis=0),
          "hi": np.percentile(b_train[FEATURES].values, hi, axis=0)}
    votes = ((val[FEATURES].values < st["lo"]) | (val[FEATURES].values > st["hi"])).sum(axis=1)
    for k in range(1, len(FEATURES) + 1):
        if (votes >= k)[y_val == 0].mean() <= STAGE1_MAX_FPR:
            st["k"], st["name"] = k, f"B: pita p{lo}-p{hi}, K={k}"
            candidates.append(st)
            break

rows = []
for st in candidates:
    m = apply_stage1(val, st)
    rows.append({"Desain": st["name"],
                 "FPR Benign": m[y_val == 0].mean(),
                 "Deteksi Serangan": m[y_val == 1].mean()})
stage1_df = pd.DataFrame(rows)
print(as_percent(stage1_df).to_string(index=False))

STAGE1 = candidates[int(stage1_df["Deteksi Serangan"].values.argmax())]
print(f"\nTahap 1 terpilih: {STAGE1['name']}")

s1_val  = apply_stage1(val,  STAGE1)
s1_test = apply_stage1(test, STAGE1)

# 4. TAHAP 2: ONE-CLASS SVM DETECTION

Model dilatih **hanya dengan data latih Benign**. Label pada data validasi dipakai sebatas
memilih `nu` dan `gamma`, tidak ikut membentuk batas wilayah normal.

In [ ]:
print("[4/7] Grid search dan pelatihan One-Class SVM...\n")

rng = np.random.default_rng(RANDOM_STATE)
X_tune = X_train[rng.choice(len(X_train), size=min(TUNE_SAMPLE, len(X_train)), replace=False)]
Xv, yv = X_val[~s1_val], y_val[~s1_val]

grid = []
for nu in NU_GRID:
    for gamma in GAMMA_GRID:
        m = OneClassSVM(kernel="rbf", nu=nu, gamma=gamma, cache_size=1000).fit(X_tune)
        s = -m.decision_function(Xv)
        grid.append({"nu": nu, "gamma": gamma,
                     "pAUC": roc_auc_score(yv, s, max_fpr=SYSTEM_MAX_FPR),
                     "ROC-AUC": roc_auc_score(yv, s)})

grid_df = pd.DataFrame(grid).sort_values(["pAUC", "ROC-AUC"], ascending=False)
grid_df.to_csv(ARTIFACT_DIR / "grid_search_ocsvm.csv", index=False)
print("5 kombinasi terbaik:")
print(grid_df.head().round(4).to_string(index=False))

BEST_NU, BEST_GAMMA = float(grid_df.iloc[0]["nu"]), float(grid_df.iloc[0]["gamma"])
t0 = time.time()
ocsvm = OneClassSVM(kernel="rbf", nu=BEST_NU, gamma=BEST_GAMMA, cache_size=1000).fit(X_train)
print(f"\nTerpilih: nu={BEST_NU}, gamma={BEST_GAMMA}")
print(f"Waktu pelatihan: {time.time() - t0:.2f} detik | support vector: {len(ocsvm.support_)}")

# Skor anomali: makin besar makin menyimpang dari trafik normal.
s2_val  = -ocsvm.decision_function(X_val)
s2_test = -ocsvm.decision_function(X_test)

**Kalibrasi ambang keputusan.** Ambang dipilih pada data validasi, bukan memakai `predict()`
yang titik potongnya tetap di nol, sehingga sensitivitas sistem dapat diatur tanpa melatih ulang model.

In [ ]:
T_HYBRID = choose_threshold(y_val, s2_val, SYSTEM_MAX_FPR, forced=s1_val)
T_OCSVM  = choose_threshold(y_val, s2_val, SYSTEM_MAX_FPR)   # untuk ablation "Tahap 2 saja"

print(f"Ambang sistem hybrid : {T_HYBRID:.5f}")
print(f"Ambang OC-SVM saja   : {T_OCSVM:.5f}")

# 5. KEPUTUSAN AKHIR HYBRID (TWO-STAGE DECISION)

Sebuah flow dinyatakan serangan apabila ditandai Tahap 1 **atau** skor anomalinya melewati ambang Tahap 2.

In [ ]:
print("[5/7] Menggabungkan keputusan Tahap 1 dan Tahap 2...\n")

pred_s1     = s1_test                        # ablasi: Tahap 1 saja
pred_s2     = s2_test >= T_OCSVM             # ablasi: Tahap 2 saja
pred_hybrid = s1_test | (s2_test >= T_HYBRID)   # sistem usulan

# Flow yang diputus Tahap 1 diberi skor maksimum agar ROC-AUC hybrid dapat dihitung.
score_hybrid = np.where(s1_test, s2_test.max() + 1, s2_test)

print(f"Diputus serangan oleh Tahap 1 : {s1_test.sum()}")
print(f"Ditambah oleh Tahap 2         : {int((pred_hybrid & ~s1_test).sum())}")
print(f"Total diprediksi serangan     : {pred_hybrid.sum()} dari {len(test)} flow uji")

# 6. EVALUASI KESELURUHAN & SAVING ARTIFACTS

*Ablation study*: setiap komponen diuji sendiri-sendiri pada data uji yang sama.
Data uji baru disentuh mulai poin ini.

In [ ]:
print("[6/7] Evaluasi performa model pada data uji:\n")

report = pd.DataFrame([
    evaluate("Ablasi A — Tahap 1 saja (tanpa OC-SVM)",    y_test, pred_s1),
    evaluate("Ablasi B — Tahap 2 saja (tanpa threshold)", y_test, pred_s2, s2_test),
    evaluate("SISTEM USULAN — Hybrid dua tahap",          y_test, pred_hybrid, score_hybrid),
])
report.to_csv(ARTIFACT_DIR / "hasil_evaluasi.csv", index=False)
print(as_percent(report).to_string(index=False))

cm = confusion_matrix(y_test, pred_hybrid)
print(f"\nConfusion matrix hybrid [[TN, FP], [FN, TP]]:\n{cm}")

**Trade-off recall terhadap FPR.** Ambang untuk setiap anggaran FPR dipilih di data validasi, lalu diterapkan ke data uji.

In [ ]:
rows = []
for b in FPR_BUDGETS:
    t_h = choose_threshold(y_val, s2_val, b, forced=s1_val)
    t_o = choose_threshold(y_val, s2_val, b)
    for name, pred in [("Tahap 2 saja", s2_test >= t_o),
                       ("Hybrid",       s1_test | (s2_test >= t_h))]:
        r = evaluate(name, y_test, pred)
        rows.append({"Target FPR": f"{b:.0%}", "Model": name,
                     "FPR uji": f"{r['FPR']*100:.2f}%", "Recall": f"{r['Recall']*100:.2f}%",
                     "Precision": f"{r['Precision']*100:.2f}%", "F1": f"{r['F1-Score']*100:.2f}%"})

tradeoff = pd.DataFrame(rows)
tradeoff.to_csv(ARTIFACT_DIR / "tradeoff_fpr_recall.csv", index=False)
print(tradeoff.to_string(index=False))

**Kontribusi Tahap 1** terhadap sistem secara keseluruhan.

In [ ]:
atk = y_test == 1
print(f"Serangan pada data uji                  : {atk.sum()}")
print(f"Ditangkap Tahap 1                       : {(s1_test & atk).sum()}")
print(f"  di antaranya terlewat oleh OC-SVM saja: {(s1_test & atk & ~pred_s2).sum()}")
print(f"Flow uji yang diputus di Tahap 1        : {s1_test.sum()} ({s1_test.mean()*100:.2f}%)")
print(f"False positive dari Tahap 1             : {(s1_test & ~atk).sum()}")

**Waktu inferensi per flow**, sebagai ukuran efisiensi arsitektur dua tahap.

In [ ]:
REPEAT = 3
n = len(test)

t0 = time.perf_counter()
for _ in range(REPEAT):
    apply_stage1(test, STAGE1)
t_s1 = (time.perf_counter() - t0) / REPEAT

lolos = test.loc[~s1_test, FEATURES]
t0 = time.perf_counter()
for _ in range(REPEAT):
    ocsvm.decision_function(preproc.transform(lolos))
t_s2 = (time.perf_counter() - t0) / REPEAT

t0 = time.perf_counter()
for _ in range(REPEAT):
    ocsvm.decision_function(preproc.transform(test[FEATURES]))
t_all = (time.perf_counter() - t0) / REPEAT

print(f"Flow diputus di Tahap 1        : {s1_test.sum()} / {n} ({s1_test.mean()*100:.2f}%)")
print(f"Tahap 1 saja                   : {t_s1 / n * 1e6:.3f} us/flow")
print(f"Hybrid dua tahap               : {(t_s1 + t_s2) / n * 1e6:.3f} us/flow")
print(f"OC-SVM tanpa penyaringan awal  : {t_all / n * 1e6:.3f} us/flow")

**Visualisasi hasil**: confusion matrix dan kurva ROC.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["Benign", "Serangan"])
ax.set_yticks([0, 1], ["Benign", "Serangan"])
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
ax.set_title("Confusion matrix — hybrid")

ax = axes[1]
for name, sc in [("Tahap 2 saja", s2_test), ("Hybrid dua tahap", score_hybrid)]:
    fpr, tpr, _ = roc_curve(y_test, sc)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, sc):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.axvline(SYSTEM_MAX_FPR, color="gray", ls=":", label=f"Batas FPR {SYSTEM_MAX_FPR:.0%}")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Kurva ROC — data uji"); ax.legend()

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "evaluasi_hybrid.png", dpi=150)
plt.show()

**Menyimpan artefak model.** Catatan: saat memuat ulang dengan `joblib.load`,
fungsi `signed_log1p` dan `apply_stage1` harus sudah didefinisikan lebih dulu.

In [ ]:
bundle = {
    "features": FEATURES,
    "stage1": STAGE1,
    "preprocessor": preproc,
    "ocsvm": ocsvm,
    "stage2_threshold": T_HYBRID,
    "config": {"scope": SCOPE, "nu": BEST_NU, "gamma": BEST_GAMMA,
               "system_max_fpr": SYSTEM_MAX_FPR},
}
joblib.dump(bundle, MODEL_DIR / "ddos_two_stage_model.pkl")
print("Model disimpan di", MODEL_DIR / "ddos_two_stage_model.pkl")

def predict_two_stage(new_df, bundle):
    """Kembalikan prediksi (0 = normal, 1 = serangan) dan tahap yang memutuskan."""
    new_df = new_df.copy()
    new_df.columns = new_df.columns.str.strip()
    new_df[bundle["features"]] = (new_df[bundle["features"]]
                                  .replace([np.inf, -np.inf], np.nan).fillna(0).astype("float64"))
    s1 = apply_stage1(new_df, bundle["stage1"])
    score = -bundle["ocsvm"].decision_function(
        bundle["preprocessor"].transform(new_df[bundle["features"]]))
    pred = s1 | (score >= bundle["stage2_threshold"])
    stage = np.where(s1, "Tahap 1", np.where(pred, "Tahap 2", "-"))
    return pred.astype(int), stage

pred_check, _ = predict_two_stage(test, bundle)
assert (pred_check == pred_hybrid.astype(int)).all()
print("Model tersimpan dan fungsi inferensi terverifikasi.")

# 7. BREAKDOWN DETEKSI PER-JENIS SERANGAN

Seluruh jenis serangan merupakan pola yang tidak pernah dilihat model saat pelatihan,
sehingga bagian ini sekaligus berfungsi sebagai uji generalisasi.

In [ ]:
print("[7/7] Hasil deteksi per jenis serangan (bahan Bab IV):\n")

ev = test[["Label", "y"]].copy()
ev["tahap1"] = s1_test
ev["prediksi"] = pred_hybrid

rows = []
for label, sub in ev[ev["y"] == 1].groupby("Label"):
    d1, dt = int(sub["tahap1"].sum()), int(sub["prediksi"].sum())
    rows.append({"Jenis Serangan": label, "Sampel Uji": len(sub),
                 "Terdeteksi Tahap 1": d1, "Terdeteksi Tahap 2": dt - d1,
                 "Total Terdeteksi": dt, "Recall (%)": round(dt / len(sub) * 100, 2)})

per_attack = pd.DataFrame(rows).sort_values("Recall (%)", ascending=False)
per_attack.to_csv(ARTIFACT_DIR / "recall_per_serangan.csv", index=False)
print(per_attack.to_string(index=False))

fp = ev[ev["y"] == 0]
print(f"\nFalse positive Benign — Tahap 1: {int(fp['tahap1'].sum())} | "
      f"Total: {int(fp['prediksi'].sum())} dari {len(fp)}")

# 8. EKSPOR ARTEFAK UNTUK SISTEM WEB

Bagian ini menyiapkan berkas pendukung yang dibaca aplikasi web:
`metrics.json` untuk halaman Dasbor serta berkas contoh untuk halaman Deteksi.

In [ ]:
print("[8/8] Menyiapkan artefak untuk sistem web...\n")

nama_ringkas = {
    "Ablasi A — Tahap 1 saja (tanpa OC-SVM)": "Tahap 1 saja",
    "Ablasi B — Tahap 2 saja (tanpa threshold)": "Tahap 2 saja",
    "SISTEM USULAN — Hybrid dua tahap": "Dua tahap (usulan)",
}

metrics = {
    "komposisi": [
        {"himpunan": "Latih",    "benign": int(len(b_train)), "serangan": 0,               "total": int(len(b_train))},
        {"himpunan": "Validasi", "benign": int(len(b_val)),   "serangan": int(len(a_val)), "total": int(len(val))},
        {"himpunan": "Uji",      "benign": int(len(b_test)),  "serangan": int(len(a_test)),"total": int(len(test))},
    ],
    "ablation": [
        {
            "nama": nama_ringkas.get(r["Model"], r["Model"]),
            "accuracy": float(r["Accuracy"]), "precision": float(r["Precision"]),
            "recall": float(r["Recall"]), "f1": float(r["F1-Score"]), "fpr": float(r["FPR"]),
            "roc_auc": None if pd.isna(r["ROC-AUC"]) else float(r["ROC-AUC"]),
            "pr_auc": None if pd.isna(r["PR-AUC"]) else float(r["PR-AUC"]),
        }
        for _, r in report.iterrows()
    ],
    "per_serangan": [
        {
            "label": r["Jenis Serangan"], "sampel": int(r["Sampel Uji"]),
            "tahap1": int(r["Terdeteksi Tahap 1"]), "tahap2": int(r["Terdeteksi Tahap 2"]),
            "recall": float(r["Recall (%)"]) / 100,
        }
        for _, r in per_attack.iterrows()
    ],
    "catatan": f"Sumber: CSE-CIC-IDS2018, {len(file_paths)} berkas harian, lingkup {SCOPE.upper()}.",
}

(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("metrics.json ditulis ke", MODEL_DIR / "metrics.json")

# Berkas contoh untuk halaman Deteksi: data uji penuh dan potongan kecil untuk demo.
kolom_ekspor = FEATURES + ["Label"]
test[kolom_ekspor].to_csv(DATA_OUT_DIR / "data_uji.csv", index=False)
(test[kolom_ekspor]
 .sample(n=min(500, len(test)), random_state=RANDOM_STATE)
 .to_csv(DATA_OUT_DIR / "data_demo.csv", index=False))

print(f"data_uji.csv  : {len(test)} flow")
print(f"data_demo.csv : {min(500, len(test))} flow")

**Pemeriksaan akhir.** Sel di bawah memuat ulang model persis seperti yang dilakukan sistem web,
lalu menjalankannya pada berkas contoh. Bila hasilnya sama dengan evaluasi di atas, artefak siap dipakai.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT))
sys.modules.pop("model_service", None)
import model_service as ms

info = ms.info_model()
print("Fitur   :", info["jumlah_fitur"])
print("Tahap 1 :", info["stage1_nama"])
print("Tahap 2 :", info["kernel"], "| nu", info["nu"], "| gamma", info["gamma"],
      "| ambang", round(info["ambang"], 5))

contoh = ms.baca_berkas(DATA_OUT_DIR / "data_demo.csv")
_, ringkas = ms.deteksi(contoh)
print("\nUji muat ulang pada data_demo.csv")
print(f"  total flow  : {ringkas['total']}")
print(f"  serangan    : {ringkas['serangan']} "
      f"(Tahap 1: {ringkas['tahap1']}, Tahap 2: {ringkas['tahap2']})")
print(f"  accuracy    : {ringkas['accuracy']*100:.2f}%")
print(f"  recall      : {ringkas['recall']*100:.2f}%")
print(f"  FPR         : {ringkas['fpr']*100:.2f}%")
print(f"  waktu       : {ringkas['durasi']} detik")
print("\nArtefak siap. Jalankan sistem web dengan: python app.py")